In [1]:
# Cell 1: Setup and Imports (Modified)
# A set of experiments designed to see how the Weighted Harmonic Mean gain metric impacts the performance of CON-FOLD.

import numpy as np
import pandas as pd
import time
import os
import sqlite3
from statistics import mean, stdev
from joblib import Parallel, delayed
from IPython.display import display

# Import your project files
# Ensure they are in the python path
from algo import metric_list # This can be kept for other experiments, but we will override it below
from foldrm import Classifier
from utils import split_data, split_xy, get_scores, count_rules_in_model, num_predicates, prune_rules, get_inverse_brier_score
from datasets import wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean

datasets = [wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean]
dataset_names = ["Wine", "Ecoli", "Weight Lifting", "Wall Robot", "Page Blocks", "Nursery", "Dry Bean"]

# --- NEW: Define the range of Beta values to test ---
beta_values = [0.5, 1.0, 1.5, 2.0]

# --- MODIFIED: Dynamically create the metric list with string names ---
# We no longer need the metric_functions dictionary
metric_list = [f"WHM_beta_{beta}" for beta in beta_values]

# Display options for the final DataFrame
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

C:\Users\Lachlan McGinness\GithubRepositories\CONFOLD\algo.py:2: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.stats import binom


In [2]:
# Cell 2: Helper functions (Corrected)
import sqlite3

def run_one_trial(dataset_name, metric_name, fit_method, data, model_template, num_classes, is_confidence, db_file):
    """
    Runs a single trial and saves the results directly to the SQLite database.
    """
    start_time = time.time()
    
    # --- MODIFIED: Prepare arguments for the fit methods ---
    fit_kwargs = {}
    metric_to_use = metric_name

    # Check if this is our special WHM metric
    if 'WHM_beta' in metric_name:
        # The actual metric name the 'gain' function expects is 'Weighted_Harmonic_Mean'
        metric_to_use = 'Weighted_Harmonic_Mean'
        # Extract the beta value from the name and add it to our keyword arguments
        beta_value = float(metric_name.split('_')[-1])
        fit_kwargs['beta'] = beta_value

    model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    data_train, data_test = split_data(data, ratio=0.8)
    X_test, Y_test = split_xy(data_test)
    
    if is_confidence:
        # Pass the metric string and any extra kwargs (like beta)
        model.confidence_fit(data_train, metric=metric_to_use, num_classes=num_classes, **fit_kwargs)
    else:
        model.fit(data_train, ratio=0.5, metric=metric_to_use, num_classes=num_classes, **fit_kwargs)
    
    Ystar_test_tuples = model.predict(X_test)
    Ystar_test = [y[0] for y in Ystar_test_tuples]
    score = get_scores(Ystar_test, data_test)
    brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
    
    rule_count = count_rules_in_model(model)
    
    model.asp()
    predicate_count = num_predicates(model)
    
    elapsed_time = time.time() - start_time
    
    # Save to database
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute('PRAGMA journal_mode=WAL;')
    cursor.execute('''
        INSERT INTO trials (dataset, metric, fit_method, accuracy, brier_score, time, num_rules, num_preds)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', (dataset_name, metric_name, fit_method, score, brier_score, elapsed_time, rule_count, predicate_count))
    conn.commit()
    conn.close()

def run_experiments_for_combination(dataset_info, metric, fit_method, remaining_trials, db_file):
    """
    Runs the remaining trials for a specific dataset, metric, and fit method in parallel.
    """
    dataset_func, name = dataset_info
    print(f"--- Starting: Dataset='{name}', Metric='{metric}', Fit Method='{fit_method}', Remaining Trials={remaining_trials} ---")
    
    model_template, data = dataset_func()
    num_classes = len(pd.unique(pd.DataFrame(data).iloc[:, -1]))
    is_confidence = (fit_method == "CON-FOLD (confidence_fit)")
    
    if remaining_trials > 0:
        Parallel(n_jobs=-1, verbose=10)(
            delayed(run_one_trial)(name, metric, fit_method, data, model_template, num_classes, is_confidence, db_file)
            for _ in range(remaining_trials)
        )
    
    print(f"--- Finished: Dataset='{name}', Metric='{metric}', Fit Method='{fit_method}' ---")

In [ ]:
# Cell 3: SETUP AND SMART RESUME WITH PROGRESSIVE SAVING VIA SQLITE
if __name__ == "__main__":
    datasets_with_names = list(zip(datasets, dataset_names))
    NUM_TRIALS = 300
    db_file = 'experiment_trials_whm.db' # Using a new DB file for these specific experiments
    fit_methods = ["FOLD-RM (fit)", "CON-FOLD (confidence_fit)"]

    # Create database and table if not exists
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS trials (
            dataset TEXT,
            metric TEXT,
            fit_method TEXT,
            accuracy REAL,
            brier_score REAL,
            time REAL,
            num_rules INTEGER,
            num_preds INTEGER
        )
    ''')
    conn.commit()

    # Get current counts for each combination
    counts = {}
    cursor.execute('''
        SELECT dataset, metric, fit_method, COUNT(*) 
        FROM trials 
        GROUP BY dataset, metric, fit_method
    ''')
    for row in cursor.fetchall():
        ds, met, fm, cnt = row
        counts[(ds, met, fm)] = cnt
    conn.close()

    # Create list of tasks to run (only remaining trials)
    tasks_to_run = []
    for dataset_info in datasets_with_names:
        name = dataset_info[1]
        # The new metric_list with beta values is used here
        for metric in metric_list:
            for fm in fit_methods:
                key = (name, metric, fm)
                current = counts.get(key, 0)
                remaining = NUM_TRIALS - current
                if remaining > 0:
                    tasks_to_run.append({
                        'dataset_info': dataset_info,
                        'metric': metric,
                        'fit_method': fm,
                        'remaining_trials': remaining,
                        'db_file': db_file
                    })

    print(f"\n>>> Total combinations with remaining trials: {len(tasks_to_run)} <<<\n")

    # Run tasks sequentially (inner parallelization handles concurrency)
    if tasks_to_run:
        for task in tasks_to_run:
            run_experiments_for_combination(**task)
        
        print("\n--- All tasks complete! You can now run the aggregation in the next cell. ---")
    else:
        print("\n--- No remaining trials to run. Everything is complete. ---")


>>> Total combinations with remaining trials: 55 <<<

--- Starting: Dataset='Wine', Metric='WHM_beta_0.5', Fit Method='CON-FOLD (confidence_fit)', Remaining Trials=300 ---

% wine dataset (178, 14)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    5.1s
[Parallel(n_jobs=-1)]: Done 268 out of 300 | elapsed:    6.0s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done 300 out 

--- Finished: Dataset='Wine', Metric='WHM_beta_0.5', Fit Method='CON-FOLD (confidence_fit)' ---
--- Starting: Dataset='Wine', Metric='WHM_beta_1.0', Fit Method='FOLD-RM (fit)', Remaining Trials=300 ---

% wine dataset (178, 14)


[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:    4.0s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    4.5s
[Parallel(n_jobs=-1)]: Done 268 out of 300 | elapsed:    5.5s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:    5.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyB

--- Finished: Dataset='Wine', Metric='WHM_beta_1.0', Fit Method='FOLD-RM (fit)' ---
--- Starting: Dataset='Wine', Metric='WHM_beta_1.0', Fit Method='CON-FOLD (confidence_fit)', Remaining Trials=300 ---

% wine dataset (178, 14)


...

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed: 28.2min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed: 33.3min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed: 59.2min
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed: 67.0min
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed: 85.8min
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed: 103.8min
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed: 123.0min
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed: 144.9min
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed: 163.2min
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed: 192.0min
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed: 217.2min
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed: 239.5min
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed: 267.4min
[Parallel(n_jobs=-1)]: Done 268 out of 300 | elapsed: 312.4min remaining: 37.3min
[Parallel(n_jobs=-1)]: Done

--- Finished: Dataset='Dry Bean', Metric='WHM_beta_1.0', Fit Method='CON-FOLD (confidence_fit)' ---
--- Starting: Dataset='Dry Bean', Metric='WHM_beta_1.5', Fit Method='FOLD-RM (fit)', Remaining Trials=300 ---

% dry bean dataset (13611, 17)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:  4.8min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:  5.4min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  9.8min
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed: 10.7min
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed: 11.9min
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed: 16.0min
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed: 20.0min
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed: 21.7min
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed: 26.1min
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed: 28.1min
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed: 32.0min
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed: 36.7min
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed: 39.9min
[Parallel(n_jobs=-1)]: Done 268 out of 300 | elapsed: 47.5min remaining:  5.7min
[Parallel(n_jobs=-1)]: Done 300 out 

--- Finished: Dataset='Dry Bean', Metric='WHM_beta_1.5', Fit Method='FOLD-RM (fit)' ---
--- Starting: Dataset='Dry Bean', Metric='WHM_beta_1.5', Fit Method='CON-FOLD (confidence_fit)', Remaining Trials=300 ---

% dry bean dataset (13611, 17)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed: 64.3min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed: 74.7min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed: 118.8min
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed: 141.7min
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed: 171.8min
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed: 205.1min
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed: 251.7min
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed: 280.6min
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed: 332.0min
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed: 362.0min
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed: 409.7min


In [ ]:
# Cell 4: This cell aggregates the trial data from the database, computes averages/std, and saves to the backup CSV.

# --- MODIFIED: New file names for this experiment ---
backup_file = 'weighted_harmonic_mean_results_backup.csv'
db_file = 'experiment_trials_whm.db'
NUM_TRIALS = 300

# 1. Load all trial data from the database
conn = sqlite3.connect(db_file)
try:
    all_trials_df = pd.read_sql_query("SELECT * FROM trials", conn)
except pd.io.sql.DatabaseError:
    all_trials_df = pd.DataFrame() 
conn.close()

if not all_trials_df.empty:
    # 2. Group by experiment combination and calculate aggregates
    agg_functions = {
        'accuracy': ['mean', 'std'],
        'brier_score': ['mean', 'std'],
        'time': ['mean'],
        'num_rules': ['mean', 'std'],
        'num_preds': ['mean', 'std'],
    }
    
    summary_df = all_trials_df.groupby(['dataset', 'metric', 'fit_method']).agg(agg_functions)
    summary_df['trial_count'] = all_trials_df.groupby(['dataset', 'metric', 'fit_method'])['accuracy'].count()

    # 3. Filter for combinations that are fully complete
    completed_df = summary_df[summary_df['trial_count'] >= NUM_TRIALS].copy()
    
    results_list = []
    if not completed_df.empty:
        # 4. Flatten the MultiIndex columns and format for saving
        completed_df.columns = ['_'.join(col).strip() for col in completed_df.columns.values]
        completed_df.reset_index(inplace=True)

        for _, row in completed_df.iterrows():
            results_list.append({
                "Dataset": row['dataset'],
                "Metric": row['metric'],
                "Fit Method": row['fit_method'],
                "Avg Accuracy": row['accuracy_mean'],
                "Std Accuracy": row['accuracy_std'] if pd.notnull(row['accuracy_std']) else 0,
                "Avg Brier Score": row['brier_score_mean'],
                "Std Brier Score": row['brier_score_std'] if pd.notnull(row['brier_score_std']) else 0,
                "Avg Time": row['time_mean'],
                "Num Rules Avg": row['num_rules_mean'],
                "Num Rules Std": row['num_rules_std'] if pd.notnull(row['num_rules_std']) else 0,
                "Num Preds Avg": row['num_preds_mean'],
                "Num Preds Std": row['num_preds_std'] if pd.notnull(row['num_preds_std']) else 0
            })

    if results_list:
        final_df = pd.DataFrame(results_list)
        final_df.to_csv(backup_file, index=False)
        print(f"--- Aggregated results for {len(final_df)} completed combinations saved to {backup_file} ---")
    else:
        print("--- No completed combinations with at least 300 trials yet. ---")
else:
    print("--- No trial data found in the database. ---")

In [ ]:
# Cell 5: Process, Display, and Save Results

def process_and_display_results(df, fit_method_name, filename):
    """
    Helper function to pivot, average, format, display, and save results.
    """
    # --- 1. Pivot the DataFrame (Datasets as Columns) ---
    pivoted_df = df.set_index(['Metric', 'Fit Method', 'Dataset'])
    pivoted_df = pivoted_df.unstack(level='Dataset')
    pivoted_df = pivoted_df.swaplevel(0, 1, axis=1)
    pivoted_df.sort_index(axis=1, level=0, inplace=True)
    
    # --- 2. Calculate the Average Across All Datasets ---
    average_df = df.groupby(['Metric', 'Fit Method']).mean(numeric_only=True)
    average_df.columns = pd.MultiIndex.from_product([['Average'], average_df.columns])
    
    # --- 3. Combine Average and Pivoted DataFrames ---
    final_summary_df = pd.concat([average_df, pivoted_df], axis=1)
    
    # --- 4. Format the DataFrame for Display ---
    format_dict = {
        "Accuracy Avg": "{:.4f}",
        "Accuracy Std": "{:.4f}",
        "Brier Score Avg": "{:.4f}",
        "Brier Score Std": "{:.4f}",
        "Avg Time": "{:.3f}s",
        "Num Rules Avg": "{:.2f}",
        "Num Rules Std": "{:.2f}",
        "Num Preds Avg": "{:.2f}",
        "Num Preds Std": "{:.2f}"
    }
    
    accuracy_cols = [col for col in final_summary_df.columns if col[1] == 'Avg Accuracy']
    brier_cols = [col for col in final_summary_df.columns if col[1] == 'Avg Brier Score']

    formatted_df = final_summary_df.style.format(format_dict) \
        .bar(subset=accuracy_cols, color='#5fba7d', vmin=0.5) \
        .bar(subset=brier_cols, color='#c9a1e8', vmin=0.5)
    
    print(f"\n--- EXPERIMENT SUMMARY: {fit_method_name} ---")
    display(formatted_df)
    
    # --- 5. Save the Summary to a CSV file ---
    final_summary_df.to_csv(filename)
    print(f"\nSummary results have been successfully saved to {filename}")


# --- Main Processing ---
# --- MODIFIED: Load from the new aggregated backup CSV ---
backup_file = 'weighted_harmonic_mean_results_backup.csv'
if os.path.exists(backup_file):
    results_df_long = pd.read_csv(backup_file)
else:
    results_df_long = pd.DataFrame()
    print("--- No aggregated results yet. Run Cell 4 after experiments. ---")

if not results_df_long.empty:
    # Process FOLD-RM (fit) results
    fold_rm_df = results_df_long[results_df_long['Fit Method'] == 'FOLD-RM (fit)'].copy()
    # --- MODIFIED: New output filename ---
    process_and_display_results(fold_rm_df, "FOLD-RM (fit)", "fold_rm_weighted_harmonic_mean_summary.csv")

    # Process CON-FOLD (confidence_fit) results
    con_fold_df = results_df_long[results_df_long['Fit Method'] == 'CON-FOLD (confidence_fit)'].copy()
    # --- MODIFIED: New output filename ---
    process_and_display_results(con_fold_df, "CON-FOLD (confidence_fit)", "con_fold_weighted_harmonic_mean_summary.csv")
else:
    print("--- No data to process yet. ---")